# Linear Wave (1D)

This notebook runs a small-amplitude 1D acoustic wave through the compressible SPH suite: a sinusoidal density/pressure perturbation propagating at the sound speed, tracked over one crossing of the periodic domain. It is a dispersion/dissipation check -- `plotState` overlays the analytic travelling wave on the sampled particles and plots the density error directly, so any phase drift or amplitude decay from the scheme shows up immediately.

Like `sod_1d.ipynb`, this notebook calls the real case code (`warpSPH.cases.linearWave.linearWaveCase`) rather than re-deriving it, and keeps the step loop unrolled in a cell instead of hiding it inside `warpSPH.runner.run()`. Plotting calls `plotState` directly (the same function `linearWaveCase.setupPlot`/`updatePlot` call internally, and the pattern the pre-`Case` version of this notebook already used) rather than going through the `Case` hooks' `openWindow`/`pumpEvents`, which does not live-update reliably inside a Jupyter cell in this environment.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/02-Linear_wave.gif)


In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float64', verbose=True)

from warpSPH import *
from warpSPH.cases.linearWave import linearWaveCase
from warpSPH.caseUtils import plotState
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import matplotlib.pyplot as plt
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `02-linear-wave.py` (or
# any other CaseSpec-driven case script), made explicit and editable here.
# `linearWaveCase.defaults`/`linearWaveCase.params` are the same values the CLI
# script starts from -- anything not overridden below just keeps its case
# default.
spec = CaseSpec(caseName=linearWaveCase.name, scheme=linearWaveCase.scheme,
                params=dict(linearWaveCase.params)) \
    .merged(**linearWaveCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=200,
    dim=1,
    L=1.0,

    # --- scheme ----------------------------------------------------------
    kernel='Wendland4',
    supportMode='Gather',

    # --- time stepping -----------------------------------------------------
    tLimit=1.0,
    # dt is left None -- linearWaveCase.initialConditions derives it below
    # from the sampled sound speed and `stepsPerCrossing` once the system
    # exists.

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- the wave's own knobs (amplitude, wavelength, sound speed, ...) ------
    params=dict(
        A=1e-6, lamda=1.0, c_s=1.0, rho0=1.0, gamma=5 / 3,
        nIters=16, stepsPerCrossing=1000,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`linearWaveCase.buildSystem` -> `sampleLinearWave`), not re-derived here.
ctx = buildContext(linearWaveCase, spec)
linearWaveCase.configureScheme(ctx)
system = linearWaveCase.buildSystem(ctx)
linearWaveCase.initialConditions(ctx, system)   # picks dt from the sound speed
spec = ctx.spec
runningState = system.initializeNewState()


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct plotState + plt.subplots(), not linearWaveCase.setupPlot -- see the
# intro cell for why. plotState calls fig.canvas.draw()/flush_events() itself.
fig = axis = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    fig, axis = plt.subplots(1, 3, figsize=(10, 5), squeeze=False)
    plotState(fig, axis, runningState, system, ctx.config, ctx.schemeConfig,
             ctx.param('rho0'), ctx.param('A'), ctx.param('lamda'), ctx.param('c_s'))
    fig.tight_layout()
    fig.savefig(os.path.join(ctx.imagePath, 'frame_00000.png'))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = linearWaveCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=linearWaveCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = linearWaveCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if fig is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        plotState(fig, axis, runningState, system, ctx.config, ctx.schemeConfig,
                 ctx.param('rho0'), ctx.param('A'), ctx.param('lamda'), ctx.param('c_s'))
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=linearWaveCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
